# 립리딩 전처리 · 학습 파이프라인 (Colab)

원본 영상을 Google Drive에 두고, 코랩 GPU로 전처리와 학습을 수행한다.

**실행 전 준비**
1. 런타임 → 런타임 유형 변경 → 하드웨어 가속기 **GPU** 선택
2. Drive에 영상 폴더 생성 후 녹화본 업로드
3. 파일명 규칙: `{화자}_{문구}_{번호}.mp4` — 예) `s01_물주세요_01.mp4`

화자가 **2명 이상**이어야 학습이 진행된다. 화자 단위로 학습·검증을 나누기 때문이다.

## 1. 환경 확인

In [ ]:
import torch

print(f"torch {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("경고: 런타임 유형을 GPU로 변경하세요.")

## 2. Drive 마운트

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. 경로 설정

`DRIVE_ROOT` 아래 `raw/`에 영상을 두면, 전처리 결과가 `processed/`에 저장된다.
Drive에 저장하므로 런타임이 끊겨도 전처리를 다시 하지 않아도 된다.

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/hanium-lipreading")
DRIVE_RAW = DRIVE_ROOT / "raw"
DRIVE_PROCESSED = DRIVE_ROOT / "processed"
DRIVE_CHECKPOINTS = DRIVE_ROOT / "checkpoints"

for folder in (DRIVE_RAW, DRIVE_PROCESSED, DRIVE_CHECKPOINTS):
    folder.mkdir(parents=True, exist_ok=True)

videos = sorted(
    p.name for p in DRIVE_RAW.glob("*") if p.suffix.lower() in (".mp4", ".avi", ".mov")
)
print(f"영상 {len(videos)}개")
for name in videos[:10]:
    print(f"  {name}")

## 4. 저장소 clone

이미 받아둔 저장소가 있으면 `git pull`로 최신 코드만 가져온다.
clone이 중간에 실패해 빈 폴더가 남은 경우에는 지우고 다시 받는다.

비공개 저장소면 `https://<TOKEN>@github.com/...` 형태로 토큰을 넣는다.
토큰은 노트북에 저장하지 말고 매번 입력한다.

**코드를 갱신한 뒤에는 런타임을 다시 시작하거나 모듈을 reload해야 반영된다.**
파이썬은 한 번 import한 모듈을 다시 읽지 않는다.

```python
import importlib
import src.ml.training.dataset, src.ml.training.train
importlib.reload(src.ml.training.dataset)
importlib.reload(src.ml.training.train)
from src.ml.training.train import train
```

In [ ]:
import os

REPO_URL = "https://github.com/HumanRhoid/hanium-lipreading.git"
BRANCH = "develop"
REPO_DIR = Path("/content/hanium-lipreading")

# clone이 중간에 실패하면 빈 폴더만 남아 다음 실행에서 git 명령이 어긋난다.
os.chdir("/content")
if (REPO_DIR / ".git").exists():
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull
else:
    !rm -rf {REPO_DIR}
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"작업 경로: {Path.cwd()}")

## 5. 의존성 설치

`uv sync`는 쓰지 않는다. `pyproject.toml`이 torch를 **CPU 전용 인덱스**로 고정하고 있어
코랩의 GPU torch가 CPU 버전으로 교체되기 때문이다. 필요한 것만 pip로 설치한다.

In [ ]:
!pip install --quiet mediapipe opencv-python wandb

import torch

print(f"설치 후 CUDA 사용 가능: {torch.cuda.is_available()}")

## 6. 얼굴 랜드마크 모델 내려받기

`face_landmarker.task`는 `.gitignore`에 제외돼 있어 저장소에 없다.

In [ ]:
LANDMARKER_URL = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
landmarker_path = REPO_DIR / "models" / "face_landmarker.task"
landmarker_path.parent.mkdir(parents=True, exist_ok=True)

if not landmarker_path.exists():
    !wget -q -O {landmarker_path} {LANDMARKER_URL}

print(f"{landmarker_path.name}: {landmarker_path.stat().st_size / 1e6:.1f} MB")

## 7. 전처리 — 영상을 .npy로 변환

Drive를 입출력으로 직접 지정한다. 이미 변환된 파일은 건너뛴다.

In [ ]:
import sys

sys.path.insert(0, str(REPO_DIR))

from src.ml.preprocess.vid2npy import run_batch

run_batch(raw_dir=DRIVE_RAW, processed_dir=DRIVE_PROCESSED)

## 8. 매니페스트 생성

`.npy` 파일명을 파싱해 라벨과 화자를 뽑아낸다.

In [ ]:
from scripts.build_manifest import build

manifest_path = DRIVE_ROOT / "manifest.csv"
build(processed_dir=DRIVE_PROCESSED, manifest_path=manifest_path)

In [ ]:
import csv
from collections import Counter

with open(manifest_path, encoding="utf-8") as manifest_file:
    rows = list(csv.DictReader(manifest_file))

print(f"클립 {len(rows)}개")
print(f"화자별: {dict(Counter(row['speaker_id'] for row in rows))}")
print(f"문구별: {dict(Counter(row['label_text'] for row in rows))}")

## 8-1. 학습 데이터를 로컬 디스크로 복사

Drive 마운트는 네트워크 파일시스템이라 매 에폭 수백 개를 원격에서 읽는다.
런타임 로컬 디스크로 옮기면 GPU가 데이터를 기다리는 시간이 줄어든다.

런타임이 끊기면 사라지므로 세션마다 다시 실행한다. 복사에 1~2분 걸린다.

In [ ]:
import shutil
import time

LOCAL_ROOT = Path("/content/data")
LOCAL_PROCESSED = LOCAL_ROOT / "processed"

started = time.time()
if not LOCAL_PROCESSED.exists():
    shutil.copytree(DRIVE_PROCESSED, LOCAL_PROCESSED)

# 매니페스트의 clip_path가 "processed/..." 라서 data_root만 바꾸면 그대로 맞는다.
TRAIN_ROOT = LOCAL_ROOT
local_count = len(list(LOCAL_PROCESSED.glob("*.npy")))
print(f"로컬 npy {local_count}개 · {time.time() - started:.0f}초")

## 9. 학습

체크포인트는 Drive에 저장되므로 런타임이 끊겨도 남는다.
학습 데이터는 `TRAIN_ROOT`(로컬 복사본)에서 읽어 I/O 대기를 줄인다.

**실험은 이 셀의 인자만 바꾸면 된다.** 저장소 코드를 고칠 필요가 없다.

- `seed` — 가중치 초기값·데이터 순서·증강을 한꺼번에 고정한다.
  같은 시드면 같은 결과가 나오므로 설정 비교의 전제가 된다
- `val_speakers` — 검증에 쓸 화자. **설정을 비교할 때는 반드시 고정한다.**
  `None`이면 화자 구성이 바뀔 때 검증 대상도 함께 바뀌어 비교가 깨진다
- `hidden_dim` / `num_layer` / `dropout` — 모델 크기와 정규화 강도
- `weight_decay` — 가중치를 작게 유지해 과적합을 억제
- `smoothing` — 최근 몇 에폭 평균으로 체크포인트를 판정할지. `1`이면 단일 에폭 최고치
- `label_smoothing` — 정답 확률을 100%로 몰지 않게 해 과신을 줄인다
- `grad_clip` — 드물게 튀는 그래디언트가 가중치를 흔드는 것을 막는다
- `ema_decay` — 가중치 이동평균으로 검증한다. 후반 진동이 완만해진다
- `augment` / `augmentation_config` — 증강 사용 여부와 강도
- `pretrained` — ImageNet 가중치로 백본을 초기화. 입력 정규화도 함께 바뀐다
- `freeze_backbone` — ResNet 층을 고정. `pretrained`와 함께 쓴다
- `amp` — bfloat16 혼합정밀도. GPU에서만 켜지고 속도가 2~3배 빨라진다
- `wandb_project` — 지정하면 실험이 웹 대시보드에 자동 기록된다

`label_smoothing` · `grad_clip` · `ema_decay`는 안정화 장치다. 셋 다 `0`을 주면
꺼진다. 체크포인트는 EMA를 켜면 평균 가중치로 저장되므로 검증 수치와 일치한다.

**시드 하나로 낸 결과는 그 자체로 성능이 아니다.** 같은 설정이라도 시드가 다르면
0.05~0.1 정도 흔들린다. 설정을 비교하거나 최종 수치를 낼 때는 시드 2~3개로
돌려 평균을 쓴다.

`pretrained=True`에 `freeze_backbone=False`면 학습률을 `3e-5`로 낮춘다. 좋은
초기값을 큰 보폭이 흐트러뜨리기 때문이다. 반대로 동결하면 움직이는 파라미터가
적어 `3e-4`까지 올려도 안정적이다.

**한 번에 하나만 바꾼다.** 두 개를 동시에 바꾸면 무엇이 효과였는지 알 수 없다.
`run_name`에 설정을 알아볼 수 있는 이름을 붙이면 나중에 비교하기 쉽다.

In [ ]:
from src.ml.preprocess.augmentation import AugmentationConfig
from src.ml.training.train import train

# 증강 강도를 조절하려면 설정을 만들어 넘긴다. None이면 기본값을 쓴다.
strong_augmentation = AugmentationConfig(
    brightness_probability=0.7,
    contrast_probability=0.7,
    rotation_probability=0.6,
    shift_probability=0.6,
    zoom_probability=0.6,
)

best_accuracy = train(
    manifest_path=manifest_path,
    data_root=TRAIN_ROOT,
    epochs=60,
    batch_size=8,
    learning_rate=1e-4,
    seed=42,  # 가중치 초기값·데이터 순서·증강을 함께 고정한다
    val_speakers=["s04"],  # 설정 비교 시 고정. None이면 seed로 무작위 선택
    checkpoint_path=DRIVE_CHECKPOINTS / "best.pt",
    num_workers=4,
    amp=True,
    hidden_dim=300,
    num_layer=2,
    dropout=0.3,
    weight_decay=0.01,
    smoothing=3,
    label_smoothing=0.1,  # 0이면 끔. 정답에 대한 과신을 줄인다
    grad_clip=1.0,  # 0이면 끔. 튀는 그래디언트를 잘라낸다
    ema_decay=0.999,  # 0이면 끔. 가중치 이동평균으로 검증한다
    augment=True,
    augmentation_config=None,  # strong_augmentation 으로 바꿔 강도 실험
    pretrained=False,  # True면 ImageNet 가중치 + ImageNet 입력 정규화
    freeze_backbone=False,  # pretrained와 함께 켜면 ResNet 층을 고정한다
    wandb_project="lipreading",
    run_name="baseline_s04",
)

## 9-1. 교차검증

검증 화자 한 명으로 재면 그 사람의 난이도에 결과가 좌우된다. 화자를 바꿔가며
전부 한 번씩 검증으로 쓰고 평균을 내면 화자 편차에 흔들리지 않는 수치가 나온다.

**시드도 함께 반복해야 한다.** 같은 설정이라도 시드가 다르면 0.05~0.1 흔들리는데,
이 폭이 화자 간 차이와 비슷해서 한 번씩만 돌리면 순위를 신뢰할 수 없다.

`SEEDS`를 늘릴수록 신뢰도가 올라가지만 학습 횟수가 화자 수만큼 곱해진다.
경향만 볼 때는 시드 하나로, 발표에 쓸 최종 수치는 셋으로 돌린다.

In [ ]:
SEEDS = [42]  # [42, 1, 7] 로 늘리면 화자마다 여러 번 돌려 편차까지 본다

speakers = sorted({row["speaker_id"] for row in rows})
print(f"화자 {speakers} · 시드 {SEEDS}")

results = {}
for speaker in speakers:
    for seed in SEEDS:
        print(f"\n{'=' * 18} {speaker} · seed {seed} {'=' * 18}")
        results[(speaker, seed)] = train(
            manifest_path=manifest_path,
            data_root=TRAIN_ROOT,
            epochs=30,
            batch_size=8,
            learning_rate=1e-4,
            seed=seed,
            val_speakers=[speaker],
            checkpoint_path=DRIVE_CHECKPOINTS / f"cv_{speaker}_seed{seed}.pt",
            num_workers=4,
            amp=True,
            hidden_dim=300,
            num_layer=2,
            dropout=0.3,
            smoothing=3,
            wandb_project="lipreading",
            run_name=f"cv_{speaker}_seed{seed}",
        )

print(f"\n{'=' * 50}")
per_speaker = {}
for speaker in speakers:
    values = [results[(speaker, s)] for s in SEEDS]
    per_speaker[speaker] = sum(values) / len(values)
    detail = " ".join(f"{v:.3f}" for v in values)
    spread = f" (폭 {max(values) - min(values):.3f})" if len(values) > 1 else ""
    print(f"  {speaker}: {per_speaker[speaker]:.3f}   [{detail}]{spread}")

overall = sum(per_speaker.values()) / len(per_speaker)
gap = max(per_speaker.values()) - min(per_speaker.values())
print(f"\n전체 평균 {overall:.3f} · 화자 간 편차 {gap:.3f}")

## 10. 체크포인트 확인

In [ ]:
checkpoint = torch.load(DRIVE_CHECKPOINTS / "best.pt", map_location="cpu")

print(f"에폭 {checkpoint['epoch']}")
print(f"클래스 {checkpoint['num_classes']}개")
print(f"검증 정확도 {checkpoint['val_accuracy']:.3f}")
print(f"최근 평균 {checkpoint['smoothed_accuracy']:.3f}")
print(
    f"모델 hidden {checkpoint['hidden_dim']} · "
    f"layer {checkpoint['num_layer']} · dropout {checkpoint['dropout']}"
)